Instalar las siguientes librerias y reiniciar el kernel

In [ ]:
#!pip install tensorflow
#!pip install keras
#!pip install ipympl

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
from matplotlib import cm
from formulaic import Formula
from sklearn import linear_model
from sklearn.metrics import mean_squared_error
from time import time

from sklearn.preprocessing import StandardScaler


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
import tensorflow as tf

# Descenso Por Gradiente

### Laboratorio de Datos, IC - FCEN - UBA - Verano 2026

Importamos el Regresor basado en Tensorflow y Keras. Ya viene implementado con Descenso por Gradiente y Descenso por 
Gradiente Estocástico. Sirve para hacer Regresión (Lineal y no Lineal) y para clasificación con Regresión Logística.

También importamos la función ``train_test_split_scale_center`` que separa los datos en entrenamiento y testeo, aplica ``MinMaxScaler`` y, opcionalmente, los
centra.

### Motivación 1

Resolver sistemas de ecuaciones *grandes* es computacionalmente costoso. Veamos un ejemplo, con un dataset sintético 
de $10000$ observaciones de $7000$ features cada una (un total de $70$ millones de valores).

Utilizamos el modelo:
$$Y = \beta_0 + \sum_{i=1}^{7000} \beta_iX_i$$

In [ ]:
# Generamos 11000 muestras aleatorias con 10000 features cada una
np.random.seed(11)
#n, m = 11000, 10000
n, m = 10000, 7000
mega_X = np.random.randn(n, m)
simulation_weights = np.random.randint(3, size=(mega_X.shape[1],1))
mega_Y = (mega_X @ simulation_weights + (np.random.randn(n,1)/10)).flatten()

# Separamos en entrenamiento y testeo
mega_X_train, mega_X_test, mega_y_train, mega_y_test = train_test_split(mega_X, mega_Y,
                                                                                     test_size=0.2,
                                                                                     random_state=21)

# Escalamos
scaler = StandardScaler()
mega_X_train = scaler.fit_transform(mega_X_train)
mega_X_test = scaler.transform(mega_X_test)

In [ ]:
print(mega_X_train.shape)
print(mega_y_train.shape)

<font color='red'>**CUIDADO AL CORRER ESTO, PUEDE EXPLOTAR**<font>

In [ ]:
# Hacemos regresion lineal con scikit-learn
model = linear_model.LinearRegression()
start = time()  # marcamos el tiempo de inicio del entrenamiento
model.fit(mega_X_train, mega_y_train)
tiempo_matricial = time() - start   # registramos el tiempo total de entrenamiento
mse_matricial = mean_squared_error(mega_y_test, model.predict(mega_X_test))
print('Tiempo total de entrenamiento: ', tiempo_matricial)
print('MSE en conjunto de testeo: ', mse_matricial)

In [ ]:
model.coef_

### Motivación 2

Tenemos un dataset con la evolución de nuevos casos diarios de COVID

In [ ]:
coro = pd.read_csv('casos_coronavirus.csv')
coro.reset_index(inplace=True)      # Reseteamos el índice para trabajar con los indices en vez de las fechas
so.Plot(data=coro, x='index', y='confirmados_Nuevos').add(so.Dot())

¿Qué tipo de función parece que sigue la evolución de los casos? ¿Podemos plantear un modelo de regresión **lineal**?

### 1. Ejemplo: Regresión Lineal con una variable predictora 

Usemos el dataset de ``inmuebles.csv`` para hacer regresión lineal del precio a partir de la superficie del inmueble:
$$precio = b + w\cdot superficie \qquad (precio \sim superficie) $$

In [ ]:
# Cargamos el dataset
data = pd.read_csv('inmuebles.csv')

# Normalizamos los datos
X = data[['superficie']]
y = data['precio']

# Separamos entrenamiento y testeo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Modelo: y = w*x + b
model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],1)),   # Definimos la capa de entrada, del tamaño de los datos
    tf.keras.layers.Dense(1)  # La siguiente capa es la función, cada variable va multiplicada por un coeficiente (es una función densa)
                              # El parámetro 1 indica que queremos que devuelva un solo valor (el resultado de hacer b + w1*x1 + w2*x2 + ...)
])

# Optimizador descenso por gradiente (estocastico) "gd"
opt = tf.keras.optimizers.SGD(learning_rate=0.05)

model.compile(
    optimizer=opt,
    loss="mse",
    metrics=["mse"]
)

# No especificamos tamaño del batch, usa todos los datos en cada época
history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_split=0.2,
    verbose=1, 
    batch_size=len(X_train)
    #batch_size=32
)

In [ ]:
# Graficamos la pérdida en entreanamiento y validación
import matplotlib.pyplot as plt
plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()


In [ ]:
weights, bias = model.layers[0].get_weights()

print("Pesos (W):")
print(weights)

print("\nBias (b):")
print(bias)

In [ ]:
# Evaluamos el MSE en el conjunto de testeo
model.evaluate(X_test.to_numpy(), y_test.to_numpy().reshape(-1, 1),        # A TensorFlow no le gustan los DataFrame de pandas 
               return_dict=True,                            # Devuelve un diccionario (por si usamos mas de una métrica)
               verbose=0,                                   # No imprima en pantalla el procedimiento de evaluación
               batch_size=32)

In [ ]:
# Si queremos imprimir los coeficientes en cada paso, agregamos una función "callback"
class PrintWeights(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        w, b = self.model.layers[0].get_weights()
        print(f"Epoch {epoch+1:3d} | w = {w.flatten()} | b = {b[0]:.6f} | loss = {logs['loss']:.6f}")

model.fit(
    X_train.to_numpy(),
    y_train.to_numpy().reshape(-1,1),
    epochs=20,
    verbose=0,
    callbacks=[PrintWeights()]
)

**Obs:** el desempeño y el resultado del algoritmo dependen de la elección de pesos y bias inciales

In [ ]:
# Tomamos valores iniciales w = 2, b = 5
model2 = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(
        1,
        kernel_initializer=tf.keras.initializers.Constant([[2]]),
        bias_initializer=tf.keras.initializers.Constant([5])
    )
])


# Optimizador "gd" (SGD en modo full-batch o mini-batch según batch_size)
opt = tf.keras.optimizers.SGD(learning_rate=0.05)

model2.compile(
    optimizer=opt,
    loss="mse",
    metrics=["mse"]
)

history = model2.fit(
    X_train.to_numpy(),
    y_train.to_numpy().reshape(-1,1),
    epochs=20,
    validation_split=0.2,
    verbose=0,
    callbacks=[PrintWeights()]   
)  

In [ ]:
# Evaluamos el MSE en el conjunto de testeo
model2.evaluate(X_test.to_numpy(), y_test.to_numpy().reshape(-1,1),        # A TensorFlow no le gustan los DataFrame de pandas 
               return_dict=True,                            # Devuelve un diccionario (por si usamos mas de una métrica)
               verbose=0,                                   # No imprima en pantalla el procedimiento de evaluación
               batch_size=len(y_test))

In [ ]:
# Plot loss (sin model.plot_loss())
import matplotlib.pyplot as plt
plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()


### 2. Ejemplo: Regresión Lineal con más de una variable predictora

Vamos a tratar de predecir el peso de los pingüinos según la longitud de su pico y la de su aleta, y su interacción:
$$body\_mass\_g = b + w_0\cdot bill\_length\_mm + w_1\cdot flipper\_length\_mm + 
w_2\cdot bill\_length\_mm\cdot flipper\_length\_mm$$
o, escrito con la notación de Wilkinson-Rogers: 
$$body\_mass\_g \sim bill\_length\_mm*flipper\_length\_mm$$

In [ ]:
# Cargamos el dataset
penguins = sns.load_dataset('penguins')
penguins.dropna(inplace=True)

# Elegimos los features, separamos en entrenamiento y testeo y normalizamos
X = penguins[['bill_length_mm', 'flipper_length_mm']]
y = penguins['body_mass_g']

Podemos que crearnos la matriz de datos con las variables que queremos y después solo cambiar la dimensión de entrada.

In [ ]:
# Le agregamos a X la variable producto (podemos hacerlo también con Formulaic)
X["interaction"] = X['bill_length_mm'] * X['flipper_length_mm']
y = penguins['body_mass_g']

In [ ]:
X

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11, )   # Semilla para datos de testeo

In [ ]:
# Escalamos
scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Tomamos valores iniciales w = 2, b = 5
model3 = tf.keras.Sequential([
    tf.keras.Input(shape=(3,)),  # Tenemos tres variables regresoras
    tf.keras.layers.Dense(
        1,  # La salida sigue siendo un solo valor (el resultado de aplicar la formula b + w1 x1 + w2 x2 + ...
        #kernel_initializer=tf.keras.initializers.Constant([[-310.05511],  [469.56445],  [500.6357] ]),
        #bias_initializer=tf.keras.initializers.Constant([4000])
    )
])

# Optimizador "gd" (SGD en modo full-batch o mini-batch según batch_size)
opt = tf.keras.optimizers.SGD(learning_rate=0.05)

model3.compile(
    optimizer=opt,
    loss="mse",
    metrics=["mse"]
)

history = model3.fit(
    X_train.to_numpy(),
    y_train.to_numpy().reshape(-1,1),
    epochs=20,
    validation_split=0.2,
    verbose=0,
    callbacks=[PrintWeights()]   
)  

In [ ]:
# Evaluamos el MSE en el conjunto de testeo
model3.evaluate(X_test.to_numpy(), y_test.to_numpy().reshape(-1,1),        # A TensorFlow no le gustan los DataFrame de pandas 
               return_dict=True,                            # Devuelve un diccionario (por si usamos mas de una métrica)
               verbose=0,                                   # No imprima en pantalla el procedimiento de evaluación
               batch_size=len(y_test))

In [ ]:
# Plot loss (sin model.plot_loss())
import matplotlib.pyplot as plt
plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()


Si el grafico empieza con valores muy altos podemos empezar con algun punto inicial mejor para ver mejor los errores.

### 3. Ejemplo: Regresión no lineal

Vamos a realizar Regresión Logística para clasificar pingüinos según su especie (Gentoo o Chinstrap) a partir de su 
peso. 

In [ ]:
# Nos quedamos solo con los pingüinos Gentoo o Chinstrap
penguins_clasif = penguins[penguins['species'].isin(['Chinstrap', 'Gentoo'])]

X = penguins_clasif[['body_mass_g']]
y = penguins_clasif['species']

# Transformamos y a un vector de 1's y 0's:
y = y.apply(lambda t: 1*(t == 'Gentoo'))

In [ ]:
# Graficamos los pesos según la especie
(
    so.Plot()
    .add(so.Dot(), x=X.iloc[:,0], y=y, color=penguins_clasif['species'])
)

In [ ]:
# separamos en entrenamiento y testeo, y escalamos

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11, )   # Semilla para datos de testeo

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Lo único que cambiamos es que aplicamos la función sigmoidea después de calcular $b + w*x$

In [ ]:
model4 = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(
        1,  # Sale un solo valor (el resultado de aplicar la formula b + w1 x1 + w2 x2 + ...
        activation="sigmoid", # al resultado de  b + w1 x1 + w2 x2 + ... le aplicamos la funcion sigmoidea 1 / (1 + e^(-x))
        #kernel_initializer=tf.keras.initializers.Constant([[-310.05511],  [469.56445],  [500.6357] ]),
        #bias_initializer=tf.keras.initializers.Constant([4000])
    )
])

# Optimizador "gd" (SGD en modo full-batch o mini-batch según batch_size)
opt = tf.keras.optimizers.SGD(learning_rate=0.05)

model4.compile(
    optimizer=opt,
    loss="binary_crossentropy",  # Cambiamos la función de pérdida
    metrics=["accuracy"]         # Cambiamos la medida del error
)

history = model4.fit(
    X_train.to_numpy(),
    y_train.to_numpy().reshape(-1,1),
    epochs=200,
    validation_split=0.2,
    verbose=0,
    #callbacks=[PrintWeights()]   
)  

In [ ]:
# Imprimimos pesos y bias obtenidos con el entrenamiento
weights, bias = model4.layers[0].get_weights()

print("Pesos (W):")
print(weights)

print("\nBias (b):")
print(bias)

In [ ]:
# Evaluamos el MSE en el conjunto de testeo
model4.evaluate(X_test.to_numpy(), y_test.to_numpy().reshape(-1,1),        # A TensorFlow no le gustan los DataFrame de pandas 
               return_dict=True,                            # Devuelve un diccionario (por si usamos mas de una métrica)
               verbose=0,                                   # No imprima en pantalla el procedimiento de evaluación
               batch_size=len(y_test))

In [ ]:
# Graficamos la evolución del BCE
plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

In [ ]:
# Calculamos la prediccion del modelo para X_test
y_prob_pred = model4.predict(X_test, verbose=0, batch_size=len(y_test)).flatten()
y_pred = (y_prob_pred > 0.5).astype('int')
y_pred

In [ ]:
# Ponemos todo en una matriz para graficar
data = X_test.copy()
data["y_prob_pred"] = y_prob_pred
data["y_pred"] = y_pred
data["y_test"] = y_test


In [ ]:
# Graficamos la clasificación
(
    so.Plot(data = data, x = "body_mass_g", y="y_test", color="y_pred")
    .add(so.Dot())
    .label(color='Clasificacion')
    .scale(color={1: 'red', 0:'black'})
    .add(so.Line(), y = "y_prob_pred")
)

### 4. Ejemplo: Descenso por Gradiente Estocástico 

Rehacemos la clasificación anterior, pero usando SGD

In [ ]:
X_train.shape

In [ ]:
# Tomamos valores iniciales w = 2, b = 5
model5 = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(
        1,  # Sale un solo valor (el resultado de aplicar la formula b + w1 x1 + w2 x2 + ...
        activation="sigmoid", # al resultado de  b + w1 x1 + w2 x2 + ... le aplicamos la funcion sigmoidea 1 / (1 + e^(-x))
        #kernel_initializer=tf.keras.initializers.Constant([[-310.05511],  [469.56445],  [500.6357] ]),
        #bias_initializer=tf.keras.initializers.Constant([4000])
    )
])

# Optimizador "gd" (SGD en modo full-batch o mini-batch según batch_size)
opt = tf.keras.optimizers.SGD(learning_rate=0.05)

model5.compile(
    optimizer=opt,
    loss="binary_crossentropy",  # Cambiamos la función de pérdida
    metrics=["accuracy"]         # Cambiamos la medida del error
)

history = model5.fit(
    X_train.to_numpy(),
    y_train.to_numpy().reshape(-1,1),
    epochs=30,
    batch_size=1,           # tamaño del batch
    validation_split=0.2,
    verbose=1,
    #callbacks=[PrintWeights()]   
)  

# El primer número en cada epoca nos dice cuantos batches ya completo sobre el total de batches

In [ ]:
# Calculamos el BCE en el conjunto de testeo
model5.evaluate(X_test.to_numpy(), y_test.to_numpy().reshape(-1,1),       # A TensorFlow no le gustan los DataFrame de pandas 
               return_dict=True,                            # Devuelve un diccionario (por si usamos mas de una métrica)
               verbose=0,                                   # No imprima en pantalla el procedimiento de evaluación
               batch_size=len(y_test))

In [ ]:
# Comparado con el modelo de Descenso por Gradiente
model4.evaluate(X_test.to_numpy(), y_test.to_numpy().reshape(-1,1),       # A TensorFlow no le gustan los DataFrame de pandas 
               return_dict=True,                            # Devuelve un diccionario (por si usamos mas de una métrica)
               verbose=0,                                   # No imprima en pantalla el procedimiento de evaluación
               batch_size=len(y_test))

In [ ]:
# Graficamos la evolución del BCE
plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

In [ ]:
# Calculamos la prediccion del modelo para X_test
y_prob_pred = model5.predict(X_test, verbose=0, batch_size=len(y_test)).flatten()
y_pred = (y_prob_pred > 0.5).astype('int')
y_pred

data = X_test.copy()
data["y_prob_pred"] = y_prob_pred
data["y_pred"] = y_pred
data["y_test"] = y_test

# Graficamos la clasificación
(
    so.Plot(data = data, x = "body_mass_g", y="y_test", color="y_pred")
    .add(so.Dot())
    .label(color='Clasificacion')
    .scale(color={1: 'red', 0:'black'})
    .add(so.Line(), y = "y_prob_pred")
)

### 5. Comparación con resolución matricial

Retomamos el ejemplo de las mega matrices

In [ ]:
mega_X_train

In [ ]:
mega_X_train.shape

In [ ]:
# Modelo: y = w*x + b
model = tf.keras.Sequential([
    tf.keras.Input(shape=(mega_X_train.shape[1],)),   # Definimos la capa de entrada, del tamaño de los datos
    tf.keras.layers.Dense(1)  # La siguiente capa es la función, cada variable va multiplicada por un coeficiente
])

# Optimizador "gd" (SGD en modo full-batch o mini-batch según batch_size)
opt = tf.keras.optimizers.SGD(learning_rate=0.1)

model.compile(
    optimizer=opt,
    loss="mse",
    metrics=["mse"]
)

start = time()
history = model.fit(
    mega_X_train, mega_y_train,
    epochs=200,
    batch_size = 1000, 
    validation_split=0.01,
    verbose=1
)
tiempo_SGD = time() - start

In [ ]:
# Graficamos la evolución del BCE
plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

In [ ]:
print('Tiempo total de entrenamiento: ', tiempo_SGD)

In [ ]:
weights, bias = model.layers[0].get_weights()

print("Pesos (W):")
print(weights)

print("\nBias (b):")
print(bias)

In [ ]:
# Comparado con el modelo de Descenso por Gradiente
bondad = model.evaluate(mega_X_test, mega_y_test,       # A TensorFlow no le gustan los DataFrame de pandas 
               return_dict=True,                            # Devuelve un diccionario (por si usamos mas de una métrica)
               verbose=0,                                   # No imprima en pantalla el procedimiento de evaluación
               batch_size=len(mega_y_test))
mse_SGD = bondad["mse"]
bondad

In [ ]:
print('Tiempo total de entrenamiento: ', tiempo_matricial)
print('MSE en conjunto de testeo: ', mse_matricial)

In [ ]:
print(f'En {100*(tiempo_SGD / tiempo_matricial):.2f}% de tiempo, SGD obtuvo una solucion con {100*(mse_SGD / mse_matricial-1):.2f}% mas error cuadratico medio')

## Ejercicio

En el dataset de nutricion, ajustar la variable de calorías en función de las demás variables, calculando los coeficientes del modelo lineal por descenso por gradiente (estocástico).

Probar distintas combinaciones entre cantidad de épocas, tamaño del batch y learning_rate y elegir la que parezca más apropiada.

In [ ]:
data = pd.read_csv('nutrition.csv')
data.dropna(inplace=True)
data.columns

In [ ]:
X = data.drop(columns=['FDC_ID', 'Item', 'Category', 'Calorias_kcal'])
y = data['Calorias_kcal']

# Separamos entrenamiento y testeo y escalamos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X